# OpenMarkX on a Colab GPU

Learn a watermark into Llama-3.2-1B with your own dataset, from the OpenMarkX run page.

1. **Runtime → Change runtime type → T4 GPU** (or better), then **Runtime → Run all**.
2. Open the link the second cell prints. The page is already connected to this notebook.
3. Upload a dataset (or use the C4 sample), pick a method and its hyperparameters, press **Start**.
4. When the job is done, press **Download model and key** on the page.

The second cell keeps running while you use the page: leave it running and keep this tab open. Colab stops a free
runtime that looks idle, and everything on it (the server, a running job, its files) goes with it.
A 300-step PRO run on 1B uses about 9 GB of GPU memory; a T4 is slower than the 3090 it was timed on (2 min of training).

In [ ]:
# 1. Install (about 2 minutes)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install "openmarkx @ git+https://github.com/UCF-ML-Research/OpenMarkX"
# a restarted session keeps the installed package: always take the latest code from GitHub
!pip -q install --force-reinstall --no-deps "openmarkx @ git+https://github.com/UCF-ML-Research/OpenMarkX"
# Colab ships torchao 0.10; recent peft refuses to build LoRA layers next to a torchao older than 0.16, and
# OpenMarkX does not use torchao
!pip -q uninstall -y torchao
# upstream MarkLLM at the pinned commit: the teachers of the markllm-*-d methods (PRO does not need it)
!git clone -q https://github.com/THU-BPM/MarkLLM.git ~/.openmarkx/MarkLLM 2>/dev/null; git -C ~/.openmarkx/MarkLLM checkout -q 112fa74
import nltk
for p in ("wordnet", "omw-1.4", "punkt", "punkt_tab", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"):
    nltk.download(p, quiet=True)

In [ ]:
# 2. Start the server and open the run page
import secrets, socket, subprocess, time
from IPython.display import HTML, display
from google.colab.output import eval_js

PORT, TOKEN = 7860, secrets.token_urlsafe(16)
server = subprocess.Popen(["openmarkx", "serve", "--port", str(PORT), "--token", TOKEN],
                          stdout=open("/content/serve.log", "w"), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        socket.create_connection(("127.0.0.1", PORT), timeout=1).close(); break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError(open("/content/serve.log").read())
url = eval_js(f"google.colab.kernel.proxyPort({PORT})").rstrip("/") + f"/#token={TOKEN}"
display(HTML(f'<a href="{url}" target="_blank" style="font-size:16px">Open the OpenMarkX run page</a>'
             f'<p style="color:#64748b">This link only works in this browser while this cell runs. Leave it running '
             f'(the stop button ends the server).</p>'))
try:                                    # a running cell keeps the runtime busy, so Colab does not reclaim it as idle
    server.wait()
except KeyboardInterrupt:
    server.terminate(); print("server stopped")